In [ ]:
size             = None
d_xyz            = None
grid_chunks      = None
file_path_NNC    = None
file_path_DTFE_Z = None
nnc              = None

In [ ]:
%run ./1___DTFE___Functions.ipynb

---
---
---

### Compute the (number) density grid for each chunk

In [ ]:
size_chunk = int(size / grid_chunks)
xyz = (np.arange(size_chunk) + 0.5) * d_xyz

In [ ]:
# iterate over all chunks
for         ix in range(grid_chunks):
    for     iy in range(grid_chunks):
        for iz in range(grid_chunks):
            file_path_DTFE_Z_chunk = file_path_DTFE_Z+"chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+"/"
            
            with open(file_path_DTFE_Z_chunk+"buffer.pk",      'rb') as f: buffer      = pkl.load(f)
            with open(file_path_DTFE_Z_chunk+"tree.pk",        'rb') as f: tree        = pkl.load(f)
            with open(file_path_DTFE_Z_chunk+"coords.pk",      'rb') as f: coords      = pkl.load(f)
            with open(file_path_DTFE_Z_chunk+"connections.pk", 'rb') as f: connections = pkl.load(f)
            with open(file_path_DTFE_Z_chunk+"rho.pk",         'rb') as f: rho         = pkl.load(f)
            Ainv_all = np.load(file_path_DTFE_Z_chunk+"Ainv_all.npy")
            origins  = np.load(file_path_DTFE_Z_chunk+"origins.npy")

            # Get the tree candidates... much faster in a single batch iff we use workers=-1 (parallelize on all available cores)
            coords_flat = np.stack(np.meshgrid(xyz+buffer, xyz+buffer, xyz+buffer, indexing="ij"),axis=-1).reshape(-1, 3)

            # To make sure the kernel doesn't implode on 16Gb RAM.
            all_candidates = np.empty((coords_flat.shape[0], nnc), dtype=np.int32)
            for i0 in range(4):
                i_start =  i0   *int(size_chunk**3/4)
                i_end   = (i0+1)*int(size_chunk**3/4)
                _, candidates = tree.query(coords_flat[i_start:i_end], k=nnc, workers=-1)
                #candidates = tree.query_ball_point(coords_flat[i_start:i_end], r=buffer/10)
                all_candidates[i_start:i_end] = candidates

            density, zero_points = compute_density_grid(coords_flat, np.array(all_candidates), origins, Ainv_all, coords, connections, rho)
            density = density.reshape(size_chunk, size_chunk, size_chunk)
            
            with open(file_path_NNC+"density_chunk___"    +str(ix)+"_"+str(iy)+"_"+str(iz)+".pk", 'wb') as f: pkl.dump(density,     f)
            with open(file_path_NNC+"zero_points_chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+".pk", 'wb') as f: pkl.dump(zero_points, f)

In [ ]:
del density, coords, connections, rho, tree, Ainv_all, origins; gc.collect()

---

### Merge all the chunk grids

In [ ]:
density_grid = np.zeros(size**3, dtype=np.float64).reshape((size, size, size))
zero_points = 0

In [ ]:
# iterate over all chunks
for         ix in range(grid_chunks):
    for     iy in range(grid_chunks):
        for iz in range(grid_chunks):

            with open(file_path_NNC+"density_chunk___"    +str(ix)+"_"+str(iy)+"_"+str(iz)+".pk", 'rb') as f: density_chunk     = pkl.load(f)
            with open(file_path_NNC+"zero_points_chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+".pk", 'rb') as f: zero_points_chunk = pkl.load(f)
            zero_points += zero_points_chunk

            ix_start = ix * size_chunk; ix_end = ix_start + size_chunk
            iy_start = iy * size_chunk; iy_end = iy_start + size_chunk
            iz_start = iz * size_chunk; iz_end = iz_start + size_chunk

            density_grid[ix_start:ix_end, iy_start:iy_end, iz_start:iz_end] = density_chunk

            # Delete all the chunks
            os.remove(file_path_NNC+"density_chunk___"    +str(ix)+"_"+str(iy)+"_"+str(iz)+".pk")
            os.remove(file_path_NNC+"zero_points_chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+".pk")

In [ ]:
with open(file_path_NNC+"/grid.pk",        'wb') as f: pkl.dump(density_grid, f)
with open(file_path_NNC+"/zero_points.pk", 'wb') as f: pkl.dump(zero_points,  f)

---
---
---